In [1]:
import mlflow 
from sklearn.linear_model import LinearRegression 
from sklearn.model_selection import GridSearchCV

import utils as ut

In [2]:

print(f"Tracking Uri: {mlflow.get_tracking_uri()}")


Tracking Uri: sqlite:///c%3A/Users/danur/OneDrive - Universidad de los andes/Documentos/Programaci%C3%B3n/Soft-Sensor/ml_model_training/mlflow.db


In [14]:
#setting experiment
mlflow.set_experiment("ecoli_fermentation_PLSModel")
mlflow.start_run(run_name="first_run_ SLRGlucose - raw data")

#loading data

df = ut.Utils().load_from_csv("../Exploratory Analysis/ecoli_fermentation_dataset.csv")

#dropping out duplicates

df_1 = ut.Utils().drop_duplicates(df)

#extracting features

X_glucose, X_acetate, y_glucose, y_acetate = ut.Utils().extracting_features(df_1)

#gridsearch and cross-validation
lr = LinearRegression() 
param_grid = {"fit_intercept": [True, False], 
              "copy_X": [True, False],
                "tol":[1e-3, 1e-4, 1e-5, 1e-6], 
              "n_jobs": [None, 1, 5, 10],
               "positive": [True, False]}
lr_estimator = GridSearchCV(lr, 
                            param_grid,
                            scoring=["neg_root_mean_squared_error", "r2"],
                            refit="neg_root_mean_squared_error",
                            cv=5)
#model fitting

lr_estimator.fit(X_glucose.reshape(-1, 1),
                  y_glucose.reshape(-1, 1))
#log parameters and metrics

mlflow.log_params(lr_estimator.best_params_)
mlflow.log_metric("nrmse", lr_estimator.best_score_)
mlflow.sklearn.log_model("SLR-Glucose",
                          lr_estimator.best_estimator_)

2026/11/23 17:49:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


TypeError: argument of type 'LinearRegression' is not iterable

In [13]:
mlflow.end_run()